# Perceptron Multicapa en MNIST
### Desde cero (NumPy) y con GPU (PyTorch)

**Asignatura:** Tópicos en Inteligencia Artificial (UNSA)  
**Basado en:** *Introduction to Neural Network - Part I* por el M.Sc. Percy Maldonado Quispe  
**Fecha:** Mayo 2026  

---

### Introducción y Contexto Histórico
Este notebook contiene dos enfoques prácticos para entrenar redes neuronales:
1. **NumPy Puro (CPU)**: Implementación desde cero del paso hacia adelante y hacia atrás para comprender la mecánica interna del entrenamiento sin usar frameworks.
2. **PyTorch (CPU + GPU)**: Uso de diferenciación automática (`autograd`) y módulos secuenciales (`torch.nn`) para demostrar el rendimiento y la facilidad de uso de un framework moderno.

De acuerdo con los conceptos en *Introduction to Neural Network - Part I*:
*   **Inspiración Biológica**: El modelo computacional abstrae el funcionamiento biológico (entradas como dendritas, pesos como sinapsis, sumador como soma y salida como axón).
*   **De McCulloch-Pitts (1943) al Perceptrón (1957)**: Transición desde computaciones lógicas binarias y rígidas a un modelo continuo con pesos adaptativos (Perceptrón simple de Rosenblatt).
*   **Redes Multicapa y Activaciones**: Para aproximar funciones no lineales y resolver problemas complejos, conectamos múltiples capas ocultas y empleamos funciones de activación no lineales (como ReLU o Sigmoide) que evitan la saturación y permiten un aprendizaje estable.

**Arquitectura del MLP:** 784 (Entrada) - 256 (Oculta 1) - 128 (Oculta 2) - 10 (Salida)


## 0. Librerias y datos

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
import torchvision

# Datos via torchvision (incluido con PyTorch, sin sklearn)
print("Descargando MNIST...")
raw = torchvision.datasets.MNIST("./data", train=True,  download=True)
_t  = torchvision.datasets.MNIST("./data", train=False, download=True)

X_train = raw.data.numpy().reshape(-1, 784) / 255.0
y_train = raw.targets.numpy()
X_test  = _t.data.numpy().reshape(-1, 784)  / 255.0
y_test  = _t.targets.numpy()

print(f"Train: {X_train.shape}  Test: {X_test.shape}")

---
## Parte 1 -- MLP desde cero con NumPy (CPU)

### Forward pass
z[l] = a[l-1] @ W[l] + b[l],   a[l] = sigmoid(z[l])

La capa de salida usa **Softmax**.

### Backward pass (regla de la cadena)
- delta_L = y_hat - y  (Softmax+CE, derivada exacta)
- delta_l = (delta_{l+1} @ W_{l+1}.T) * a_l * (1 - a_l)
- dW_l = a_{l-1}.T @ delta_l / m


In [ ]:
class MLP_NumPy:
    """MLP con forward/backward explicito. Sin frameworks."""

    def __init__(self, dims):
        # Xavier init: escala por sqrt(1/fan_in)
        self.W = [np.random.randn(dims[i], dims[i+1]) * np.sqrt(1/dims[i])
                  for i in range(len(dims)-1)]
        self.b = [np.zeros((1, dims[i+1]))
                  for i in range(len(dims)-1)]

    @staticmethod
    def sigmoid(z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

    @staticmethod
    def softmax(z):
        e = np.exp(z - z.max(axis=1, keepdims=True))
        return e / e.sum(axis=1, keepdims=True)

    def forward(self, X):
        self.a = [X]
        for i, (W, b) in enumerate(zip(self.W, self.b)):
            z = self.a[-1] @ W + b
            is_last = (i == len(self.W) - 1)
            self.a.append(self.softmax(z) if is_last else self.sigmoid(z))
        return self.a[-1]

    def cross_entropy(self, y_pred, y_true):
        Y = np.eye(10)[y_true]
        return -np.mean(np.sum(Y * np.log(y_pred + 1e-8), axis=1))

    def backward(self, y_true, lr):
        m     = len(y_true)
        Y     = np.eye(10)[y_true]
        delta = self.a[-1] - Y
        for i in reversed(range(len(self.W))):
            dW = self.a[i].T @ delta / m
            db = delta.mean(axis=0, keepdims=True)
            if i > 0:
                da    = delta @ self.W[i].T
                delta = da * self.a[i] * (1 - self.a[i])
            self.W[i] -= lr * dW
            self.b[i] -= lr * db

    def fit(self, X, y, epochs=20, batch_size=256, lr=0.1, X_val=None, y_val=None):
        hist = {"loss": [], "acc": [], "val_acc": []}
        m    = len(X)
        for epoch in range(1, epochs+1):
            idx = np.random.permutation(m)
            batch_losses = []
            for start in range(0, m, batch_size):
                sl    = idx[start:start+batch_size]
                y_hat = self.forward(X[sl])
                batch_losses.append(self.cross_entropy(y_hat, y[sl]))
                self.backward(y[sl], lr)
            loss = np.mean(batch_losses)
            acc  = self.accuracy(X, y)
            hist["loss"].append(loss)
            hist["acc"].append(acc)
            if X_val is not None:
                va = self.accuracy(X_val, y_val)
                hist["val_acc"].append(va)
                print(f"Epoca {epoch:2d}  loss={loss:.4f}  acc={acc:.4f}  val_acc={va:.4f}")
            else:
                print(f"Epoca {epoch:2d}  loss={loss:.4f}  acc={acc:.4f}")
        return hist

    def accuracy(self, X, y):
        return np.mean(np.argmax(self.forward(X), axis=1) == y)

In [ ]:
np.random.seed(42)
mlp_np = MLP_NumPy([784, 256, 128, 10])

t0 = time.time()
hist_np = mlp_np.fit(
    X_train, y_train,
    epochs=20, batch_size=256, lr=0.1,
    X_val=X_test, y_val=y_test
)
t_numpy = time.time() - t0
print(f"Tiempo NumPy (CPU): {t_numpy:.1f}s")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(hist_np["loss"], "b-o", markersize=4)
ax1.set(title="Perdida (Cross-Entropy)", xlabel="Epoca", ylabel="Loss")
ax1.grid(alpha=.3)
ax2.plot(hist_np["acc"],     "b-o", markersize=4, label="Train")
ax2.plot(hist_np["val_acc"], "r-o", markersize=4, label="Test")
ax2.set(title="Accuracy", xlabel="Epoca")
ax2.grid(alpha=.3); ax2.legend()
plt.suptitle("MLP NumPy (CPU)", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

---
## Parte 2 -- MLP con PyTorch (CPU + GPU)

La misma arquitectura, con diferenciacion automatica y soporte GPU.

**5 pasos del loop de entrenamiento:**
1. `zero_grad()` limpiar gradientes acumulados
2. `forward()` propagacion hacia adelante
3. `criterion()` calcular perdida
4. `backward()` autograd calcula todos los gradientes
5. `step()` actualizar pesos con el optimizador


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
def make_loader(X, y, batch_size=256, shuffle=True):
    tx = torch.tensor(X, dtype=torch.float32)
    ty = torch.tensor(y, dtype=torch.long)
    return DataLoader(TensorDataset(tx, ty), batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(X_train, y_train)
test_loader  = make_loader(X_test,  y_test, shuffle=False)

In [ ]:
class MLP_PyTorch(nn.Module):
    """Misma arquitectura que MLP_NumPy, con autograd y soporte GPU."""

    def __init__(self, dims):
        super().__init__()
        capas = []
        for i in range(len(dims)-1):
            capas.append(nn.Linear(dims[i], dims[i+1]))
            if i < len(dims) - 2:
                capas.append(nn.Sigmoid())
        # Ultima capa lineal: CrossEntropyLoss incluye Softmax internamente
        self.red = nn.Sequential(*capas)

    def forward(self, x):
        return self.red(x)

In [ ]:
def entrenar_pytorch(model, loader_train, loader_test, epochs=20, lr=0.1):
    model     = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    hist      = {"loss": [], "acc": [], "val_acc": []}

    for epoch in range(1, epochs+1):
        model.train()
        epoch_loss, correct, total = 0.0, 0, 0
        for Xb, yb in loader_train:
            Xb, yb = Xb.to(device), yb.to(device)
            optimizer.zero_grad()         # 1. limpiar gradientes
            y_hat = model(Xb)             # 2. forward
            loss  = criterion(y_hat, yb)  # 3. perdida
            loss.backward()               # 4. backward (autograd)
            optimizer.step()              # 5. actualizar pesos
            epoch_loss += loss.item() * len(yb)
            correct    += (y_hat.argmax(1) == yb).sum().item()
            total      += len(yb)

        model.eval()
        vc, vt = 0, 0
        with torch.no_grad():
            for Xb, yb in loader_test:
                Xb, yb = Xb.to(device), yb.to(device)
                vc += (model(Xb).argmax(1) == yb).sum().item()
                vt += len(yb)

        l, a, va = epoch_loss/total, correct/total, vc/vt
        hist["loss"].append(l); hist["acc"].append(a); hist["val_acc"].append(va)
        print(f"Epoca {epoch:2d}  loss={l:.4f}  acc={a:.4f}  val_acc={va:.4f}")

    return hist

In [ ]:
device = torch.device("cpu")
print(f"=== Entrenando en: {device} ===")
torch.manual_seed(42)
model_cpu = MLP_PyTorch([784, 256, 128, 10])
t0 = time.time()
hist_cpu = entrenar_pytorch(model_cpu, train_loader, test_loader)
t_cpu = time.time() - t0
print(f"Tiempo CPU: {t_cpu:.1f}s")

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"=== GPU: {torch.cuda.get_device_name(0)} ===")
    torch.manual_seed(42)
    model_gpu = MLP_PyTorch([784, 256, 128, 10])
    t0 = time.time()
    hist_gpu = entrenar_pytorch(model_gpu, train_loader, test_loader)
    t_gpu = time.time() - t0
    print(f"Tiempo GPU: {t_gpu:.1f}s  |  Speedup vs CPU: {t_cpu/t_gpu:.1f}x")
else:
    print("GPU no disponible.")
    hist_gpu, t_gpu = None, None

---
## Parte 3 -- Comparacion final

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, metric, title in zip(axes, ["loss", "val_acc"], ["Perdida", "Accuracy (Test)"]):
    ax.plot(hist_np[metric],  "b-o", markersize=4, label="NumPy (CPU)")
    ax.plot(hist_cpu[metric], "g-s", markersize=4, label="PyTorch (CPU)")
    if hist_gpu:
        ax.plot(hist_gpu[metric], "r-^", markersize=4, label="PyTorch (GPU)")
    ax.set(title=title, xlabel="Epoca"); ax.grid(alpha=.3); ax.legend()
plt.suptitle("NumPy vs PyTorch CPU vs PyTorch GPU", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

k = "val_acc"
sep = "=" * 50
print(sep)
print(f"  NumPy   (CPU)  acc={hist_np[k][-1]:.4f}  t={t_numpy:.1f}s")
print(f"  PyTorch (CPU)  acc={hist_cpu[k][-1]:.4f}  t={t_cpu:.1f}s")
if hist_gpu:
    print(f"  PyTorch (GPU)  acc={hist_gpu[k][-1]:.4f}  t={t_gpu:.1f}s  speedup={t_cpu/t_gpu:.1f}x")
print(sep)

---
## Visualizacion: ejemplos mal clasificados

In [ ]:
y_pred_np = np.argmax(mlp_np.forward(X_test), axis=1)
errores   = np.where(y_pred_np != y_test)[0]

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for ax, idx in zip(axes.flat, errores[:16]):
    ax.imshow(X_test[idx].reshape(28, 28), cmap="gray")
    ax.set_title(f"R:{y_test[idx]} P:{y_pred_np[idx]}", fontsize=8)
    ax.axis("off")
plt.suptitle("Ejemplos mal clasificados (NumPy MLP)", fontsize=12)
plt.tight_layout(); plt.show()